In [1]:
import pickle

In [2]:
# only_query_MSE = "/workspace/InstructIR/eval/model_pred/InstructIR/corpus/for_only_query_test/only_queries/only_query/_workspace_fixed_promptriever-1B_vanilla_v8-idea3-MSE-clip_checkpoint-7616.pickle"
# only_query_vanilla = "/workspace/InstructIR/eval/model_pred/InstructIR/corpus/for_only_query_test/only_queries/only_query/_workspace_promptriever-1B_vanilla_v8_no_weight_checkpoint-7616.pickle"

# inst_query_MSE = "/workspace/InstructIR/eval/model_pred/InstructIR/corpus/test/queries/query_first_eos_token/_workspace_fixed_promptriever-1B_vanilla_v8-idea3-MSE-clip_checkpoint-7616.pickle"
# inst_query_vanilla = "/workspace/InstructIR/eval/model_pred/InstructIR/corpus/test/queries/query_first_eos_token/_workspace_promptriever-1B_vanilla_v8_no_weight_checkpoint-7616.pickle"

In [3]:
import numpy as np
from scipy import stats

In [3]:
from collections import defaultdict 
import numpy as np 
from typing import List, Dict, Tuple
import logging
from beir.retrieval.evaluation import EvaluateRetrieval
import pytrec_eval
from scipy import stats

logger = logging.getLogger(__name__)

class CustomEvaluateRetrieval(EvaluateRetrieval):
    @staticmethod
    def robustness_evaluate(
                queries: Dict[str, Dict[str, int]],
                qrels: Dict[str, Dict[str, int]], 
                results: Dict[str, Dict[str, float]], 
                k_values: List[int],
                type='query',
                ignore_identical_ids: bool=True) -> Tuple[Dict[str, float], Dict[str, float], Dict[str, List[float]]]:
        
        if ignore_identical_ids:
            logging.info('For evaluation, we ignore identical query and document ids (default), please explicitly set ``ignore_identical_ids=False`` to ignore this.')
            popped = []
            for qid, rels in results.items():
                for pid in list(rels):
                    if qid == pid:
                        results[qid].pop(pid)
                        popped.append(pid)

        metrics = defaultdict(list)
        group_metric = {}
        group_scores_per_group = {}  # 각 그룹별 점수 저장
        
        # 통계 검정을 위한 상세 데이터 저장
        detailed_scores = {
            'individual_ndcg': {},  # 개별 NDCG 값들 (t-test용)
            'group_min_ndcg': {}     # 그룹별 최저 NDCG 값 (Wilcoxon test용)
        }
        
        for k in k_values:
            group_metric[f"NDCG@{k}"] = []
            group_metric[f"MAP@{k}"] = []
            group_metric[f"Recall@{k}"] = []
            group_metric[f"P@{k}"] = []
            
            group_metric[f"NDCG@{k}_min"] = []
            group_metric[f"MAP@{k}_min"] = []
            group_metric[f"Recall@{k}_min"] = []
            group_metric[f"P@{k}_min"] = []
            
            group_scores_per_group[f"NDCG@{k}"] = []
            group_scores_per_group[f"MAP@{k}"] = []
            group_scores_per_group[f"Recall@{k}"] = []
            group_scores_per_group[f"P@{k}"] = []
            
            # 상세 데이터 초기화
            detailed_scores['individual_ndcg'][f"NDCG@{k}"] = []
            detailed_scores['group_min_ndcg'][f"NDCG@{k}"] = []
        
        map_string = "map_cut." + ",".join([str(k) for k in k_values])
        ndcg_string = "ndcg_cut." + ",".join([str(k) for k in k_values])
        recall_string = "recall." + ",".join([str(k) for k in k_values])
        precision_string = "P." + ",".join([str(k) for k in k_values])
            
        # Grouping instances based on [instruction / query]
        group_per_instruction = {}
        group_per_query = {}
        for qid, rels in results.items():
            query_info = queries[qid]
            instruction, query = query_info.split('[SEP]')
            instruction = instruction.strip()
            query = query.strip()

            if not group_per_instruction.get(instruction):
                group_per_instruction[instruction] = {}

            if not group_per_query.get(query):
                group_per_query[query] = {}
                    
            group_per_instruction[instruction][qid] = rels
            group_per_query[query][qid] = rels

        if type == 'instruction':
            logger.info("Group per instruction is selected")
            target_group = group_per_instruction
        elif type == 'query':
            logger.info("Group per query is selected")
            target_group = group_per_query
        else:
            raise NotImplementedError
                
        logger.info(f"Total {len(target_group)} number of group is selected out of {len(results)}")

        for group_id, group_results in target_group.items():
            ndcg = {}
            _map = {}
            recall = {}
            precision = {}
            
            for k in k_values:
                ndcg[f"NDCG@{k}"] = []
                _map[f"MAP@{k}"] = []
                recall[f"Recall@{k}"] = []
                precision[f"P@{k}"] = []
            
            evaluator = pytrec_eval.RelevanceEvaluator(qrels, {map_string, ndcg_string, recall_string, precision_string})
            group_scores = evaluator.evaluate(group_results)
            
            for query_id in group_scores.keys():
                for k in k_values:
                    ndcg[f"NDCG@{k}"].append(group_scores[query_id]["ndcg_cut_" + str(k)])
                    _map[f"MAP@{k}"].append(group_scores[query_id]["map_cut_" + str(k)])
                    recall[f"Recall@{k}"].append(group_scores[query_id]["recall_" + str(k)])
                    precision[f"P@{k}"].append(group_scores[query_id]["P_"+ str(k)])

            # 각 그룹의 평균 점수 저장
            for k in k_values:
                group_scores_per_group[f"NDCG@{k}"].append(np.mean(ndcg[f"NDCG@{k}"]))
                group_scores_per_group[f"MAP@{k}"].append(np.mean(_map[f"MAP@{k}"]))
                group_scores_per_group[f"Recall@{k}"].append(np.mean(recall[f"Recall@{k}"]))
                group_scores_per_group[f"P@{k}"].append(np.mean(precision[f"P@{k}"]))
                
                # 통계 검정용 데이터 저장
                # 1. 개별 NDCG 값들 (모든 query에 대한 개별 점수)
                detailed_scores['individual_ndcg'][f"NDCG@{k}"].extend(ndcg[f"NDCG@{k}"])
                
                # 2. 그룹별 최저 NDCG 값
                detailed_scores['group_min_ndcg'][f"NDCG@{k}"].append(min(ndcg[f"NDCG@{k}"]))
                
                # 기존 메트릭 계산
                group_metric[f"NDCG@{k}_min"].append(min(ndcg[f"NDCG@{k}"]))
                group_metric[f"NDCG@{k}"].extend(ndcg[f"NDCG@{k}"])
                group_metric[f"MAP@{k}_min"].append(min(_map[f"MAP@{k}"]))
                group_metric[f"MAP@{k}"].extend(_map[f"MAP@{k}"])
                group_metric[f"Recall@{k}_min"].append(min(recall[f"Recall@{k}"]))
                group_metric[f"Recall@{k}"].extend(recall[f"Recall@{k}"])
                group_metric[f"P@{k}_min"].append(min(precision[f"P@{k}"]))
                group_metric[f"P@{k}"].extend(precision[f"P@{k}"])
        
        # 평균 및 통계 계산
        statistics = {}
        for name, v in group_metric.items():
            metrics['cluster_{}_avg'.format(name)] = sum(v) / len(v)
        
        # 그룹 간 통계 검정
        for k in k_values:
            for metric_name in [f"NDCG@{k}", f"MAP@{k}", f"Recall@{k}", f"P@{k}"]:
                scores = group_scores_per_group[metric_name]
                
                # 기본 통계량
                statistics[f'{metric_name}_mean'] = np.mean(scores)
                statistics[f'{metric_name}_std'] = np.std(scores)
                statistics[f'{metric_name}_min'] = np.min(scores)
                statistics[f'{metric_name}_max'] = np.max(scores)
                statistics[f'{metric_name}_median'] = np.median(scores)
                
                # 그룹이 2개 이상일 때만 통계 검정 수행
                if len(scores) >= 2:
                    # 정규성 검정 (Shapiro-Wilk test)
                    if len(scores) >= 3:
                        _, p_value_normality = stats.shapiro(scores)
                        statistics[f'{metric_name}_normality_p'] = p_value_normality
                    
                    # 그룹 간 분산 계산 (변동성 측정)
                    statistics[f'{metric_name}_cv'] = np.std(scores) / np.mean(scores) if np.mean(scores) > 0 else 0

        metrics = dict(sorted(metrics.items()))
        statistics = dict(sorted(statistics.items()))

        logging.info("\n")
        logging.info("=" * 80)
        logging.info(f"Group per {type}")
        logging.info("=" * 80)
        
        logging.info("\nAverage Metrics:")
        for k in metrics.keys():
            logging.info("{}: {:.4f}".format(k, metrics[k] * 100))
        
        logging.info("\n" + "=" * 80)
        logging.info("Group-level Statistics:")
        logging.info("=" * 80)
        for k in statistics.keys():
            if 'normality_p' in k or 'cv' in k:
                logging.info("{}: {:.4f}".format(k, statistics[k]))
            else:
                logging.info("{}: {:.4f}".format(k, statistics[k] * 100))
        
        logging.info("\n" + "=" * 80)
        logging.info("Detailed Scores Summary (for statistical testing):")
        logging.info("=" * 80)
        for k in k_values:
            individual_count = len(detailed_scores['individual_ndcg'][f"NDCG@{k}"])
            group_count = len(detailed_scores['group_min_ndcg'][f"NDCG@{k}"])
            logging.info(f"NDCG@{k}: {individual_count} individual scores, {group_count} group min scores")

        return metrics, statistics, detailed_scores
    
    @staticmethod
    def compare_models(
        detailed_scores_model1: Dict[str, Dict[str, List[float]]],
        detailed_scores_model2: Dict[str, Dict[str, List[float]]],
        k_values: List[int],
        model1_name: str = "Model 1",
        model2_name: str = "Model 2"
    ) -> Dict[str, Dict[str, float]]:
        """
        두 모델 간의 통계적 유의성 검정
        
        Args:
            detailed_scores_model1: 첫 번째 모델의 detailed_scores
            detailed_scores_model2: 두 번째 모델의 detailed_scores
            k_values: k 값 리스트
            model1_name: 첫 번째 모델 이름
            model2_name: 두 번째 모델 이름
            
        Returns:
            comparison_results: 통계 검정 결과
        """
        comparison_results = {}
        
        logging.info("\n" + "=" * 80)
        logging.info(f"Statistical Comparison: {model1_name} vs {model2_name}")
        logging.info("=" * 80)
        
        for k in k_values:
            metric_key = f"NDCG@{k}"
            
            # 1. 개별 NDCG 값에 대한 t-test (paired)
            individual_scores_1 = detailed_scores_model1['individual_ndcg'][metric_key]
            individual_scores_2 = detailed_scores_model2['individual_ndcg'][metric_key]
            
            # 길이가 같은지 확인 (paired test를 위해)
            if len(individual_scores_1) == len(individual_scores_2):
                t_stat, t_pvalue = stats.ttest_rel(individual_scores_1, individual_scores_2)
                comparison_results[f'{metric_key}_individual_ttest_statistic'] = t_stat
                comparison_results[f'{metric_key}_individual_ttest_pvalue'] = t_pvalue
                
                mean_diff = np.mean(individual_scores_1) - np.mean(individual_scores_2)
                comparison_results[f'{metric_key}_individual_mean_diff'] = mean_diff
                
                logging.info(f"\n{metric_key} - Individual Scores (Paired t-test):")
                logging.info(f"  {model1_name} mean: {np.mean(individual_scores_1):.4f}")
                logging.info(f"  {model2_name} mean: {np.mean(individual_scores_2):.4f}")
                logging.info(f"  Mean difference: {mean_diff:.4f}")
                logging.info(f"  t-statistic: {t_stat:.4f}")
                logging.info(f"  p-value: {t_pvalue:.4f}")
                logging.info(f"  Significant (α=0.05): {'Yes' if t_pvalue < 0.05 else 'No'}")
            else:
                logging.warning(f"{metric_key}: Individual scores have different lengths. Using unpaired t-test.")
                t_stat, t_pvalue = stats.ttest_ind(individual_scores_1, individual_scores_2)
                comparison_results[f'{metric_key}_individual_ttest_statistic'] = t_stat
                comparison_results[f'{metric_key}_individual_ttest_pvalue'] = t_pvalue
                comparison_results[f'{metric_key}_individual_mean_diff'] = np.mean(individual_scores_1) - np.mean(individual_scores_2)
            
            # 2. 그룹별 최저 성능에 대한 Wilcoxon signed-rank test
            group_min_scores_1 = detailed_scores_model1['group_min_ndcg'][metric_key]
            group_min_scores_2 = detailed_scores_model2['group_min_ndcg'][metric_key]
            
            if len(group_min_scores_1) == len(group_min_scores_2):
                wilcoxon_stat, wilcoxon_pvalue = stats.wilcoxon(group_min_scores_1, group_min_scores_2)
                comparison_results[f'{metric_key}_groupmin_wilcoxon_statistic'] = wilcoxon_stat
                comparison_results[f'{metric_key}_groupmin_wilcoxon_pvalue'] = wilcoxon_pvalue
                
                median_diff = np.median(group_min_scores_1) - np.median(group_min_scores_2)
                comparison_results[f'{metric_key}_groupmin_median_diff'] = median_diff
                
                logging.info(f"\n{metric_key} - Group Min Scores (Wilcoxon signed-rank test):")
                logging.info(f"  {model1_name} median: {np.median(group_min_scores_1):.4f}")
                logging.info(f"  {model2_name} median: {np.median(group_min_scores_2):.4f}")
                logging.info(f"  Median difference: {median_diff:.4f}")
                logging.info(f"  Wilcoxon statistic: {wilcoxon_stat:.4f}")
                logging.info(f"  p-value: {wilcoxon_pvalue:.4f}")
                logging.info(f"  Significant (α=0.05): {'Yes' if wilcoxon_pvalue < 0.05 else 'No'}")
            else:
                logging.warning(f"{metric_key}: Group min scores have different lengths. Cannot perform paired Wilcoxon test.")
        
        logging.info("\n" + "=" * 80)
        
        return comparison_results

In [4]:
from beir.datasets.data_loader import GenericDataLoader
import os

corpus, queries, qrels = GenericDataLoader(
        data_folder='/workspace/InstructIR/INSTRUCTIR',
        corpus_file='corpus.jsonl',
        qrels_file= os.path.join('/workspace/InstructIR/INSTRUCTIR', 'qrels', 'test' + '.tsv'),
        query_file='queries.jsonl',
        ).load_custom()

/opt/conda/envs/instructir/lib/python3.11/site-packages/beir/datasets/data_loader.py:2: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm
  0%|          | 0/16072 [00:00<?, ?it/s]

100%|██████████| 16072/16072 [00:00<00:00, 399107.50it/s]


In [11]:
inst_query_MSE = "/workspace/InstructIR/eval/model_pred/InstructIR/corpus/test/queries/query_first_eos_token/_workspace_fixed_promptriever-1B_vanilla_v8-idea3-MSE-clip_checkpoint-7616.pickle"
inst_query_vanilla = "/workspace/InstructIR/eval/model_pred/InstructIR/corpus/test/queries/query_first_eos_token/_workspace_fixed_promptriever-1B_vanilla_v8-idea3-MSE_rep_checkpoint-7616.pickle"

In [12]:
# with open(only_query_MSE, "rb") as f:
#     only_query_MSE_dict = pickle.load(f)

# with open(only_query_vanilla, "rb") as f:
#     only_query_vanilla_dict = pickle.load(f)

with open(inst_query_MSE, "rb") as f:
    inst_query_MSE_dict = pickle.load(f)

with open(inst_query_vanilla, "rb") as f:
    inst_query_vanilla_dict = pickle.load(f)


In [13]:
# 모델 1 평가
metrics1, stats1, detailed1 = CustomEvaluateRetrieval.robustness_evaluate(
    queries, qrels, inst_query_MSE_dict, k_values=[10], type='query'
)

# 모델 2 평가
metrics2, stats2, detailed2 = CustomEvaluateRetrieval.robustness_evaluate(
    queries, qrels, inst_query_vanilla_dict, k_values=[10], type='query'
)

# 두 모델 비교
comparison = CustomEvaluateRetrieval.compare_models(
    detailed1, detailed2, k_values=[10],
    model1_name="Model A", model2_name="Model B"
)

In [14]:
comparison

{'NDCG@10_individual_ttest_statistic': 3.7762865936214243,
 'NDCG@10_individual_ttest_pvalue': 0.00016011476581282252,
 'NDCG@10_individual_mean_diff': 0.0036423289793315883,
 'NDCG@10_groupmin_wilcoxon_statistic': 55491.0,
 'NDCG@10_groupmin_wilcoxon_pvalue': 0.038559416326960735,
 'NDCG@10_groupmin_median_diff': 0.0}